In [2]:
#Load Data and Create Calendar features 

import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import os

# Load the clean dataset
df = pd.read_csv("../data/processed/clean_demand_data.csv")
df['Actual data of'] = pd.to_datetime(df['Actual data of'])
df.set_index('Actual data of', inplace=True)
df = df.sort_index()

target_col = 'Max. Demand at eve. peak (Generation end)'

# Calendar Features 
df['DayOfWeek'] = df.index.dayofweek
df['Month'] = df.index.month
df['IsWeekend'] = df['DayOfWeek'].isin([4, 5]).astype(int) 

# Lag Features
df['Demand_Lag1'] = df[target_col].shift(1)
df['Demand_Lag2'] = df[target_col].shift(2)
df['Demand_Lag7'] = df[target_col].shift(7)

# 7-day rolling average of demand
df['Demand_Rolling7_Mean'] = df[target_col].shift(1).rolling(window=7).mean()

# Dropping the first 7 rows since they now have NaNs from the lag calculations
df = df.dropna()
print(f"Dataset shape after creating lags: {df.shape}")

Dataset shape after creating lags: (1809, 47)


In [3]:
#Dimensionality Reduction 

# Isolate regional demand columns
regional_demand_cols = [col for col in df.columns if 'demand' in col.lower() and col != target_col]

# Standardize before PCA
scaler = StandardScaler()
scaled_regional = scaler.fit_transform(df[regional_demand_cols])

# Applying PCA and keeping only the first principla component 
pca = PCA(n_components=1) 
df['Regional_Demand_PCA'] = pca.fit_transform(scaled_regional)

variance_explained = pca.explained_variance_ratio_[0] * 100
print(f"Variance explained by 1st Principal Component: {variance_explained:.2f}%")
print("Report Note: The first PCA component captures the vast majority of regional demand variance, effectively reducing dimensionality while retaining the core demand signal.")

Variance explained by 1st Principal Component: 70.56%
Report Note: The first PCA component captures the vast majority of regional demand variance, effectively reducing dimensionality while retaining the core demand signal.


In [4]:
#Feature Selection & Export

# Keeping the target,engineered features, and the supply constraints
features_to_keep = [
    target_col,
    'DayOfWeek', 'Month', 'IsWeekend',
    'Demand_Lag1', 'Demand_Lag2', 'Demand_Lag7', 'Demand_Rolling7_Mean',
    'Gas/LF limitation', 'Coal supply Limitation', 'Low water level in Kaptai lake',
    'Regional_Demand_PCA'
]

# Keeping load columns for Question 3, if we need them for localized error analysis later
load_cols = [col for col in df.columns if 'load' in col.lower()]
final_columns = list(set(features_to_keep + load_cols))

df_featured = df[final_columns]


df_featured.to_csv("../data/processed/featured_dataset.csv")
print("Featured dataset saved successfully!")
print(f"Final feature count: {df_featured.shape[1] - 1}") # -1 for target

Featured dataset saved successfully!
Final feature count: 20
